In [ ]:
# =========================
# RQ6: Robustness and Generalization
# Binary Classification Version
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# -------------------------
# 1. Load Data
# -------------------------
df = pd.read_csv(
    "/kaggle/input/datasets/sharmajicoder/gaming-and-mental-health/gaming_mental_health_10M_40features.csv"
)

df = df.dropna()
df = df.sample(n=20000, random_state=42)

TARGET = df.columns[-1]

# -------------------------
# 2. Encode Target + Convert to Binary
# -------------------------
le = LabelEncoder()
y_raw = le.fit_transform(df[TARGET])

median_value = np.median(y_raw)
y = (y_raw > median_value).astype(int)

print("Class distribution:")
print(pd.Series(y).value_counts())

# -------------------------
# 3. Features
# -------------------------
X = pd.get_dummies(df.drop(TARGET, axis=1), drop_first=True).astype(float)

# -------------------------
# 4. Model
# -------------------------
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

# -------------------------
# 5. Train/Test Split
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scenarios = []
results = []

def evaluate(name, y_true, y_pred):
    scenarios.append(name)
    results.append([
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, average="weighted", zero_division=0),
        recall_score(y_true, y_pred, average="weighted", zero_division=0),
        f1_score(y_true, y_pred, average="weighted", zero_division=0)
    ])

# -------------------------
# 6. Scenario 1: Baseline
# -------------------------
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

evaluate("Train/Test (80/20)", y_test, y_pred)

# -------------------------
# 7. Scenario 2: 5-Fold CV
# -------------------------
cv_acc = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
).mean()

scenarios.append("5-Fold CV")
results.append([cv_acc, cv_acc, cv_acc, cv_acc])

# -------------------------
# 8. Scenario 3: +10% Noise
# -------------------------
np.random.seed(42)

X_test_noise = X_test.copy()
noise = np.random.normal(0, 0.1, X_test_noise.shape)
X_test_noise = X_test_noise + noise

model.fit(X_train, y_train)
y_pred_noise = model.predict(X_test_noise)

evaluate("+10% Noise", y_test, y_pred_noise)

# -------------------------
# 9. Scenario 4: 20% Missing Data
# -------------------------
np.random.seed(42)

X_test_missing = X_test.copy()
mask = np.random.rand(*X_test_missing.shape) < 0.2
X_test_missing = X_test_missing.mask(mask)

X_test_missing = X_test_missing.fillna(X_train.mean())

model.fit(X_train, y_train)
y_pred_missing = model.predict(X_test_missing)

evaluate("20% Missing Data", y_test, y_pred_missing)

# -------------------------
# 10. Results Table
# -------------------------
df_results = pd.DataFrame(
    results,
    columns=["Accuracy", "Precision", "Recall", "F1"]
)

df_results["Scenario"] = scenarios
df_results.to_csv("RQ6_table.csv", index=False)

print("\n=== RQ6 Results ===")
print(df_results)

# -------------------------
# 11. Plot
# -------------------------
plt.figure(figsize=(9, 6))

metrics = ["Accuracy", "Precision", "Recall", "F1"]
markers = ["o", "s", "^", "D"]

for i, metric in enumerate(metrics):
    values = df_results[metric]

    plt.plot(
        df_results["Scenario"],
        values,
        marker=markers[i],
        linewidth=2,
        label=metric
    )

    for j, val in enumerate(values):
        plt.text(j, val + 0.01, f"{val:.2f}", ha="center", fontsize=9)

plt.title("RQ6: Robustness and Generalization", fontsize=14, fontweight="bold")
plt.xlabel("Experimental Scenario", fontsize=12, fontweight="bold")
plt.ylabel("Score", fontsize=12, fontweight="bold")

plt.ylim(0.0, 1.05)

plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("RQ6_figure.pdf", bbox_inches="tight")
plt.show()